<a href="https://colab.research.google.com/github/memo124/Laboratorio_IA_etica/blob/main/Laboratorio_IA_etica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Carga y Exploración de Datos

En esta sección importamos las librerías esenciales y cargamos el dataset del Titanic. Cada fila representa a un pasajero individual del navío, y nuestra columna objetivo o variable a predecir es **Survived** (donde `1` significa que sobrevivió y `0` que no).

In [ ]:
import pandas as pd
import numpy as np
import os

# Buscamos el archivo de forma local; si no está, lo descargamos automáticamente
file_path = 'train.csv'
if not os.path.exists(file_path):
    print("[INFO] 'train.csv' no se encontró localmente. Descargándolo desde la fuente pública...")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.to_csv(file_path, index=False)
else:
    print("[INFO] Cargando 'train.csv' desde el almacenamiento local...")
    df = pd.read_csv(file_path)

# Una mirada rápida para entender qué pinta tienen los datos
print("=== VISTA PREVIA DE LOS PASAJEROS ===")
display(df.head())

# Revisamos tipos de datos y la presencia de valores nulos
print("\n=== INFORMACIÓN GENERAL DEL DATASET ===")
df.info()

print("\n=== CANTIDAD DE VALORES FALTANTES POR COLUMNA ===")
print(df.isna().sum())

[INFO] Cargando 'train.csv' desde el almacenamiento local...
=== VISTA PREVIA DE LOS PASAJEROS ===


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



=== INFORMACIÓN GENERAL DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

=== CANTIDAD DE VALORES FALTANTES POR COLUMNA ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0


## Paso 2: División Estratégica del Dataset

Para garantizar una evaluación justa y evitar sesgos, dividimos el conjunto de datos original en tres partes utilizando una distribución estratificada respecto a la supervivencia:
1. **Entrenamiento (60%)**: Para que los algoritmos aprendan los patrones.
2. **Validación (20%)**: Para experimentar, tunear hiperparámetros y comparar alternativas.
3. **Prueba (20%)**: Reservado exclusivamente para la evaluación final.

In [ ]:
from sklearn.model_selection import train_test_split

# 'Survived' es la etiqueta que queremos predecir; el resto son características
X = df.drop(columns=['Survived'])
y = df['Survived']

# Primero aislamos el 20% para el test final.
# Usamos stratify para asegurarnos de que la proporción de sobrevivientes se mantenga.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Del 80% restante, tomamos el 25% para validación (que equivale al 20% del total original)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print("=== REPARTO DE LOS CONJUNTOS DE DATOS ===")
print(f"Entrenamiento (X_train): {X_train.shape} | ({round(len(X_train)/len(df)*100)}% del total)")
print(f"Validación    (X_val):   {X_val.shape} | ({round(len(X_val)/len(df)*100)}% del total)")
print(f"Prueba        (X_test):  {X_test.shape} | ({round(len(X_test)/len(df)*100)}% del total)")

=== REPARTO DE LOS CONJUNTOS DE DATOS ===
Entrenamiento (X_train): (534, 11) | (60% del total)
Validación    (X_val):   (178, 11) | (20% del total)
Prueba        (X_test):  (179, 11) | (20% del total)


## Paso 3: Preprocesamiento de Datos Básicos

Construimos un pipeline para tratar de manera limpia los datos numéricos y categóricos. Para evitar fugas de información, calculamos las reglas de imputación (mediana para numéricos, moda para categóricos) **únicamente** con el set de entrenamiento, y luego transformamos validación y prueba bajo estas mismas reglas.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Columnas seleccionadas para iniciar nuestro modelo
selected_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train_filtered = X_train[selected_cols]
X_val_filtered = X_val[selected_cols]
X_test_filtered = X_test[selected_cols]

# Separamos variables por su naturaleza para aplicar diferentes transformaciones
num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Pclass', 'Sex', 'Embarked']

# Pipeline numérico: Imputamos valores ausentes con la mediana
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Pipeline categórico: Imputamos con el más común y aplicamos codificación OneHot
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Integramos ambos procesos en un único transformador de columnas
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# ¡MUY IMPORTANTE! Ajustamos (fit) solo con entrenamiento para no contaminar el proceso
X_train_prepared = preprocessor.fit_transform(X_train_filtered)
X_val_prepared = preprocessor.transform(X_val_filtered)
X_test_prepared = preprocessor.transform(X_test_filtered)

# Recuperamos el nombre de las columnas generadas para mantener la claridad
cat_features = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)
feature_names = num_cols + list(cat_features)

print("=== PROCESO DE PREPARACIÓN COMPLETADO ===")
print(f"Total de columnas resultantes: {len(feature_names)}")
print(f"Nombres de las columnas: {feature_names}")

=== PROCESO DE PREPARACIÓN COMPLETADO ===
Total de columnas resultantes: 12
Nombres de las columnas: ['Age', 'SibSp', 'Parch', 'Fare', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


## Paso 4: Entrenamiento y Evaluación del Modelo Baseline

Entrenamos un clasificador `RandomForestClassifier` estándar como base de comparación. Evaluamos su desempeño en el conjunto de validación mediante la métrica **F1-Score**, que servirá como nuestra línea de base (baseline) para medir futuras mejoras.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# Creamos el modelo de referencia con una semilla para reproducibilidad
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train_prepared, y_train)

# Realizamos predicciones sobre el conjunto de validación que apartamos previamente
y_val_pred = baseline_model.predict(X_val_prepared)

# Medimos el rendimiento del modelo baseline
val_f1 = f1_score(y_val, y_val_pred)

print("=== RESULTADOS DEL MODELO BASELINE ===")
print(f"F1-Score en Validación: {val_f1:.4f}")
print("\nReporte Detallado de Métricas de Clasificación:")
print(classification_report(y_val, y_val_pred))

=== RESULTADOS DEL MODELO BASELINE ===
F1-Score en Validación: 0.7704

Reporte Detallado de Métricas de Clasificación:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       110
           1       0.78      0.76      0.77        68

    accuracy                           0.83       178
   macro avg       0.82      0.81      0.82       178
weighted avg       0.83      0.83      0.83       178

